In [0]:
%sql
CREATE OR REPLACE TABLE dataengineering.swift.dim_defect (
    defect_code STRING,
    description STRING,
    remediation_group STRING
)

In [0]:
%sql
INSERT INTO dataengineering.swift.dim_defect VALUES
('CLEAN',               'Address meets structured requirements',        'None'),
  ('MISSING_COUNTRY',     'Country field absent',                         'Auto-fixable'),
  ('PLACEHOLDER_COUNTRY', 'Country contains placeholder text',            'Auto-fixable'),
  ('INVALID_COUNTRY',     'Country code not in ISO 3166',                 'Auto-fixable'),
  ('DUPLICATED_TOWN',     'Town appears in both structured and free text','Auto-fixable'),
  ('MISSING_TOWN',        'Town field absent',                            'Needs source data'),
  ('UNSTRUCTURED_ONLY',   'Address held entirely as free text',           'Needs source data')

In [0]:
%sql
SELECT * FROM dataengineering.swift.dim_defect

In [0]:
%sql
CREATE OR REPLACE TABLE dataengineering.swift.dim_corridor
( corridors STRING,
  country_name STRING,
  region STRING,
  infra_status STRING

)

In [0]:
%sql
INSERT INTO dataengineering.swift.dim_corridor VALUES
  ('US', 'United States',  'Americas',           'Migrated'),
  ('GB', 'United Kingdom', 'Europe',             'Migrated'),
  ('IN', 'India',          'Southern Asia',      'Unknown'),
  ('PH', 'Philippines',    'Southeast Asia',     'Unknown'),
  ('PK', 'Pakistan',       'Southern Asia',      'Unknown'),
  ('CN', 'China',          'Eastern Asia',       'Unknown'),
  ('MX', 'Mexico',         'Americas',           'Unknown'),
  ('NG', 'Nigeria',        'Sub-Saharan Africa', 'Unknown')


In [0]:
%sql
CREATE OR REPLACE TABLE dataengineering.swift.dim_channel
( channel STRING,
  segment STRING,
  format STRING,
  migration_status STRING
);

INSERT INTO dataengineering.swift.dim_channel VALUES
  ('PAIN001_SCORE_PLUS', 'Corporate', 'ISO 20022',   'Native'),
  ('MT101_SCORE',        'Corporate', 'MT',          'Legacy'),
  ('PROPRIETARY',        'Corporate', 'Bank portal', 'Native'),
  ('RETAIL_APP',         'Retail',    'Bank app',    'Native'),
  ('BRANCH',             'Retail',    'Staff entry', 'Native')

In [0]:
%sql
SELECT COUNT(*) FROM(SELECT uetr,creation_dt,amount_cad,corridor,channel,debtor_defect,creditor_defect FROM dataengineering.swift.silver_payments 
UNION ALL
SELECT uetr,creation_dt,amount_cad,corridor,channel,debtor_defect,creditor_defect FROM dataengineering.swift.silver_quarantine)

In [0]:
%sql
CREATE OR REPLACE TABLE dataengineering.swift.fct_payment AS
SELECT
  uetr,
  creation_dt,
  amount_cad,
  corridor,
  channel,
  debtor_defect,
  creditor_defect,
  CASE WHEN debtor_defect != 'CLEAN' OR creditor_defect != 'CLEAN'
       THEN TRUE ELSE FALSE END AS is_failing
FROM dataengineering.swift.silver_payments
UNION ALL
SELECT
  uetr, creation_dt, amount_cad, corridor, channel, debtor_defect, creditor_defect,
  CASE WHEN debtor_defect != 'CLEAN' OR creditor_defect != 'CLEAN'
       THEN TRUE ELSE FALSE END AS is_failing
FROM dataengineering.swift.silver_quarantine

In [0]:
%sql
SELECT is_failing, COUNT(*) AS payments FROM dataengineering.swift.fct_payment GROUP BY is_failing

In [0]:
%sql
SELECT
  ROUND(SUM(amount_cad) / 1000000, 1)                              AS total_value_m,
  ROUND(SUM(CASE WHEN is_failing THEN amount_cad ELSE 0 END) / 1000000, 1) AS failing_value_m
FROM dataengineering.swift.fct_payment;

In [0]:
%sql
SELECT * FROM dataengineering.swift.fct_payment LIMIT 10

In [0]:
%sql
SELECT COUNT(*) AS total_payments,
SUM(CASE WHEN is_failing THEN 1 ELSE 0 END) AS failing_payments,
ROUND(100.0 * SUM(CASE WHEN is_failing THEN 1 ELSE 0 END) / COUNT(*),1) AS pct_failing
FROM dataengineering.swift.fct_payment

In [0]:
%sql
CREATE OR REPLACE TABLE dataengineering.swift.dim_corridor (
  corridor     STRING,
  country_name STRING,
  region       STRING,
  infra_status STRING
);

INSERT INTO dataengineering.swift.dim_corridor VALUES
  ('US', 'United States',  'Americas',           'Migrated'),
  ('GB', 'United Kingdom', 'Europe',             'Migrated'),
  ('IN', 'India',          'Southern Asia',      'Unknown'),
  ('PH', 'Philippines',    'Southeast Asia',     'Unknown'),
  ('PK', 'Pakistan',       'Southern Asia',      'Unknown'),
  ('CN', 'China',          'Eastern Asia',       'Unknown'),
  ('MX', 'Mexico',         'Americas',           'Unknown'),
  ('NG', 'Nigeria',        'Sub-Saharan Africa', 'Unknown');

In [0]:
%sql
SELECT
  f.corridor,
  d.country_name,
  COUNT(*) AS payments,
  SUM(CASE WHEN f.is_failing THEN 1 ELSE 0 END) AS failing,
  ROUND(100.0 * SUM(CASE WHEN f.is_failing THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_failing
FROM dataengineering.swift.fct_payment f
JOIN dataengineering.swift.dim_corridor d
  ON f.corridor = d.corridor
GROUP BY f.corridor, d.country_name
ORDER BY pct_failing DESC

In [0]:
%sql
SELECT COUNT(*) FROM (SELECT corridor, 'debtor' AS party, debtor_defect AS defect FROM dataengineering.swift.fct_payment
UNION ALL
SELECT corridor, 'creditor' AS party, creditor_defect AS defect FROM dataengineering.swift.fct_payment)

In [0]:
%sql
WITH addresses AS (
  SELECT corridor, 'debtor' AS party, debtor_defect AS defect FROM dataengineering.swift.fct_payment
  UNION ALL
  SELECT corridor, 'creditor' AS party, creditor_defect AS defect FROM dataengineering.swift.fct_payment
)
SELECT
  corridor,
  defect,
  COUNT(*) AS addresses
FROM addresses
WHERE defect != 'CLEAN'
GROUP BY corridor, defect
ORDER BY corridor, addresses DESC

In [0]:
%sql
WITH addresses AS (
  SELECT corridor, debtor_defect   AS defect FROM dataengineering.swift.fct_payment
  UNION ALL
  SELECT corridor, creditor_defect AS defect FROM dataengineering.swift.fct_payment
)
SELECT
  a.corridor,
  d.remediation_group,
  COUNT(*) AS addresses
FROM addresses a
JOIN dataengineering.swift.dim_defect d
  ON a.defect = d.defect_code
WHERE d.remediation_group != 'None'
GROUP BY a.corridor, d.remediation_group
ORDER BY a.corridor, addresses DESC

In [0]:
%sql
SELECT
  f.channel,
  c.segment,
  c.migration_status,
  COUNT(*) AS payments,
  SUM(CASE WHEN f.is_failing THEN 1 ELSE 0 END) AS failing,
  ROUND(100.0 * SUM(CASE WHEN f.is_failing THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_failing
FROM dataengineering.swift.fct_payment f
JOIN dataengineering.swift.dim_channel c
  ON f.channel = c.channel
GROUP BY f.channel, c.segment, c.migration_status
ORDER BY pct_failing DESC

In [0]:
%sql
SELECT
  f.corridor,
  d.country_name,
  ROUND(SUM(f.amount_cad) / 1000000, 1) AS total_value_m,
  ROUND(SUM(CASE WHEN f.is_failing THEN f.amount_cad ELSE 0 END) / 1000000, 1) AS failing_value_m,
  ROUND(100.0 * SUM(CASE WHEN f.is_failing THEN f.amount_cad ELSE 0 END) / SUM(f.amount_cad), 1) AS pct_value_failing
FROM dataengineering.swift.fct_payment f
JOIN dataengineering.swift.dim_corridor d
  ON f.corridor = d.corridor
GROUP BY f.corridor, d.country_name
ORDER BY failing_value_m DESC

In [0]:
%sql
CREATE OR REPLACE VIEW dataengineering.swift.rpt_headline AS
SELECT
  COUNT(*) AS total_payments,
  SUM(CASE WHEN is_failing THEN 1 ELSE 0 END) AS failing_payments,
  ROUND(100.0 * SUM(CASE WHEN is_failing THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_failing
FROM dataengineering.swift.fct_payment;

CREATE OR REPLACE VIEW dataengineering.swift.rpt_corridor AS
SELECT
  f.corridor, d.country_name, d.region, d.infra_status,
  COUNT(*) AS payments,
  SUM(CASE WHEN f.is_failing THEN 1 ELSE 0 END) AS failing,
  ROUND(100.0 * SUM(CASE WHEN f.is_failing THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_failing,
  ROUND(SUM(f.amount_cad) / 1000000, 1) AS total_value_m,
  ROUND(SUM(CASE WHEN f.is_failing THEN f.amount_cad ELSE 0 END) / 1000000, 1) AS failing_value_m
FROM dataengineering.swift.fct_payment f
JOIN dataengineering.swift.dim_corridor d ON f.corridor = d.corridor
GROUP BY f.corridor, d.country_name, d.region, d.infra_status;

CREATE OR REPLACE VIEW dataengineering.swift.rpt_channel AS
SELECT
  f.channel, c.segment, c.format, c.migration_status,
  COUNT(*) AS payments,
  SUM(CASE WHEN f.is_failing THEN 1 ELSE 0 END) AS failing,
  ROUND(100.0 * SUM(CASE WHEN f.is_failing THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_failing
FROM dataengineering.swift.fct_payment f
JOIN dataengineering.swift.dim_channel c ON f.channel = c.channel
GROUP BY f.channel, c.segment, c.format, c.migration_status;

CREATE OR REPLACE VIEW dataengineering.swift.vw_address AS
SELECT uetr, corridor, channel, amount_cad, 'debtor' AS party, debtor_defect AS defect
FROM dataengineering.swift.fct_payment
UNION ALL
SELECT uetr, corridor, channel, amount_cad, 'creditor' AS party, creditor_defect AS defect
FROM dataengineering.swift.fct_payment;

CREATE OR REPLACE VIEW dataengineering.swift.rpt_corridor_defect AS
SELECT corridor, defect, COUNT(*) AS addresses
FROM dataengineering.swift.vw_address
WHERE defect != 'CLEAN'
GROUP BY corridor, defect;

CREATE OR REPLACE VIEW dataengineering.swift.rpt_remediation AS
SELECT
  a.corridor, d.remediation_group,
  COUNT(*) AS addresses
FROM dataengineering.swift.vw_address a
JOIN dataengineering.swift.dim_defect d ON a.defect = d.defect_code
WHERE d.remediation_group != 'None'
GROUP BY a.corridor, d.remediation_group;

In [0]:
%sql
SELECT * FROM dataengineering.swift.rpt_headline

In [0]:
%sql
USE CATALOG dataengineering;
SHOW VIEWS IN swift;

In [0]:
answer = (spark.read.format("csv")
    .option("header", "true")
    .load("s3://swift-pipeline-aws/Ref_data/answer_key.csv"))

answer.write.mode("overwrite").saveAsTable("dataengineering.swift.ref_answer_key")

print(answer.count())

In [0]:
%sql
SELECT COUNT(*) AS matched_rows
FROM dataengineering.swift.ref_answer_key k
JOIN dataengineering.swift.vw_address a
  ON k.uetr = a.uetr AND k.party = a.party

In [0]:
%sql
SELECT
  COUNT(*) AS total_addresses,
  SUM(CASE WHEN k.defect = a.defect THEN 1 ELSE 0 END) AS agree,
  SUM(CASE WHEN k.defect != a.defect THEN 1 ELSE 0 END) AS disagree,
  ROUND(100.0 * SUM(CASE WHEN k.defect = a.defect THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_agree
FROM dataengineering.swift.ref_answer_key k
JOIN dataengineering.swift.vw_address a
  ON k.uetr = a.uetr AND k.party = a.party

In [0]:
%sql
CREATE OR REPLACE VIEW dataengineering.swift.rpt_ground_truth AS
SELECT
  COUNT(*) AS total_addresses,
  SUM(CASE WHEN k.defect = a.defect THEN 1 ELSE 0 END) AS agree,
  SUM(CASE WHEN k.defect != a.defect THEN 1 ELSE 0 END) AS disagree,
  ROUND(100.0 * SUM(CASE WHEN k.defect = a.defect THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_agree
FROM dataengineering.swift.ref_answer_key k
JOIN dataengineering.swift.vw_address a
  ON k.uetr = a.uetr AND k.party = a.party;